# Conversi?n de voz cantada

Este cuaderno separa una interpretaci?n, convierte su timbre con Seed-VC y mezcla el resultado con el acompa?amiento. Usa una grabaci?n propia o una fuente que tengas permiso de procesar.

```text
Canci?n permitida ? voz + instrumental ? conversi?n de timbre ? mezcla WAV
```

La melod?a y el ritmo proceden de la interpretaci?n de entrada; el modelo no crea una nueva composici?n.


## Preparar un entorno independiente

Seed-VC usa versiones de dependencias distintas de las del cuaderno de s?ntesis. Se recomienda Python 3.10 y un entorno virtual separado. Instala `requirements-canto.txt` y descarga el c?digo de Seed-VC en una carpeta `seed-vc/` junto a este cuaderno. La celda de instalaci?n comprueba la carpeta y la descarga si a?n no existe.


In [ ]:
!pip install -r requirements-canto.txt


In [ ]:
from pathlib import Path
import subprocess

SEED_VC = Path("seed-vc")
if not (SEED_VC / "seed_vc_wrapper.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Plachtaa/seed-vc.git", str(SEED_VC)], check=True)
else:
    print("Seed-VC ya est? disponible en", SEED_VC)


In [ ]:
import logging
import subprocess
import sys
import time
import warnings
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import torchaudio
from IPython.display import Audio, Markdown, display

warnings.filterwarnings("ignore")
for nombre in ("transformers", "huggingface_hub"):
    logging.getLogger(nombre).setLevel(logging.ERROR)

FS = 44_100                                # frecuencia de trabajo: la de un CD
TRABAJO = Path("audios_canto")             # aquí se guarda todo lo que generemos
TRABAJO.mkdir(exist_ok=True)
SEED_VC = Path("seed-vc")

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
PASOS = 30 if DISPOSITIVO != "cpu" else 10         # pasos de difusión: más pasos, más calidad y más espera


def escuchar(audio, titulo=None, frecuencia=FS):
    """Reproduce un audio. Vale para mono (una fila) y para estéreo (dos filas)."""
    if titulo:
        display(Markdown(f"**{titulo}** · {np.asarray(audio).shape[-1] / frecuencia:.1f} s"))
    display(Audio(audio, rate=frecuencia))


print("PyTorch", torch.__version__, "· se va a usar:", DISPOSITIVO, "· pasos de difusión:", PASOS)
print("Carpeta de Seed-VC:", "encontrada" if (SEED_VC / "seed_vc_wrapper.py").exists() else "NO ESTÁ. Repasa la instalación de arriba.")

## Elegir la fuente musical

Coloca un WAV autorizado en `datos/audio/cancion_entrada.wav`. Tambi?n puedes cambiar `CANCION` en la celda de configuraci?n por otra ruta local. En equipos sin GPU, empieza con un fragmento corto.


In [ ]:
CANCION = Path("datos/audio/cancion_entrada.wav")
INICIO = 0                  # segundo en el que empieza el fragmento
DURACION = None             # segundos que dura; None = hasta el final. Sin GPU, pon 12

cancion, _ = librosa.load(CANCION, sr=FS, mono=False, offset=INICIO, duration=DURACION)
if cancion.ndim == 1:                              # si es mono, la duplicamos en los dos canales
    cancion = np.stack([cancion, cancion])
sf.write(TRABAJO / "01_fragmento.wav", cancion.T, FS)

escuchar(cancion, "La canción original")

## Separar voz e instrumental

Demucs estima dos pistas a partir de la mezcla original. La separaci?n puede dejar restos de voz o instrumentos: comprueba ambas pistas antes de convertirlas.


In [ ]:
inicio = time.time()
proceso = subprocess.run([sys.executable, "-m", "demucs", "--two-stems", "vocals", "-n", "htdemucs",
                          "-o", str(TRABAJO / "separado"), str(TRABAJO / "01_fragmento.wav")],
                         capture_output=True, text=True)
if proceso.returncode != 0:
    print(proceso.stderr[-1500:])

pistas = TRABAJO / "separado" / "htdemucs" / "01_fragmento"
voz_original, _ = librosa.load(pistas / "vocals.wav", sr=FS, mono=True)
instrumental, _ = librosa.load(pistas / "no_vocals.wav", sr=FS, mono=False)
sf.write(TRABAJO / "02_voz_original.wav", voz_original, FS)

print(f"Separado en {time.time() - inicio:.0f} s")
escuchar(voz_original, "La voz, sola")
escuchar(instrumental, "El instrumental")

Escucha la voz aislada y el acompa?amiento. Si hay artefactos fuertes, prueba otro fragmento o una fuente m?s limpia; la conversi?n posterior no corrige una separaci?n deficiente.


In [ ]:
MI_VOZ = Path("audios/02_estilo_expresivo.wav")
MI_VOZ.parent.mkdir(parents=True, exist_ok=True)
SEGUNDOS = 15
IMPORTAR = None                # por ejemplo "grabacion_del_movil.mp3"; None para usar o grabar mi_voz.wav

if IMPORTAR:
    audio, frecuencia = librosa.load(IMPORTAR, sr=None, mono=True)
    sf.write(MI_VOZ, audio, frecuencia)
    print(f"Importado {IMPORTAR} → {MI_VOZ}")

elif MI_VOZ.exists():
    print(f"Se usa {MI_VOZ}, la grabación del primer tutorial.")

else:
    import sounddevice as sd

    microfono = sd.query_devices(kind="input")
    frecuencia = int(microfono["default_samplerate"])
    print("Micrófono:", microfono["name"])
    for cuenta in ["3", "2", "1"]:
        print(cuenta, end=" … ", flush=True)
        time.sleep(1)
    print("¡HABLA!")
    grabacion = sd.rec(int(SEGUNDOS * frecuencia), samplerate=frecuencia, channels=1, dtype="float32")
    for quedan in range(SEGUNDOS, 0, -1):
        print(quedan, end=" ", flush=True)
        time.sleep(1)
    sd.wait()
    sf.write(MI_VOZ, grabacion[:, 0], frecuencia)
    print(f"\nGuardado en {MI_VOZ}")

mi_voz, fs_mi_voz = librosa.load(MI_VOZ, sr=None, mono=True)
escuchar(mi_voz, "Tu voz, la referencia", fs_mi_voz)

## Preparar el modelo

El c?digo se carga desde la carpeta local `seed-vc/`. En la primera ejecuci?n tambi?n pueden descargarse pesos del modelo; revisa el espacio disponible y la licencia del proyecto.


In [ ]:
sys.path.insert(0, str(SEED_VC.resolve()))
from seed_vc_wrapper import SeedVCWrapper

inicio = time.time()
motor = SeedVCWrapper(device=torch.device(DISPOSITIVO))

# Un ajuste necesario en los Mac con chip de Apple, e inofensivo en los demás: el detector de tono
# devuelve números de 64 bits y ese tipo de procesador solo trabaja con 32.
_detectar_tono = motor.rmvpe.infer_from_audio
motor.rmvpe.infer_from_audio = lambda *args, **kwargs: _detectar_tono(*args, **kwargs).astype("float32")

# Al cargarse, Seed-VC cambia la manera de dibujar de matplotlib. La devolvemos al cuaderno.
%matplotlib inline

print(f"\nSeed-VC cargado en {time.time() - inicio:.0f} s")

## Convertir el timbre

La funci?n aplica Seed-VC a la voz separada usando la muestra de referencia. Por defecto se reutiliza el WAV generado por el cuaderno Chatterbox; si no est?, el flujo solicita una grabaci?n local. Los audios no se guardan en Git.


In [ ]:
def convertir(fuente, referencia, semitonos=0, pasos=PASOS, guardar=None):
    """Devuelve la voz de `fuente` cantando con el timbre de `referencia`. Las dos son rutas a archivos de audio."""
    conversion = motor.convert_voice(str(fuente), str(referencia), diffusion_steps=pasos, f0_condition=True,
                                     auto_f0_adjust=False, pitch_shift=semitonos, stream_output=False)
    try:
        while True:                       # Seed-VC trabaja por trozos; los recorremos hasta que entrega el audio completo
            next(conversion)
    except StopIteration as final:
        audio = np.asarray(final.value, dtype="float32")
    if guardar:
        sf.write(TRABAJO / guardar, audio, FS)
    return audio


inicio = time.time()
voz_convertida = convertir(TRABAJO / "02_voz_original.wav", MI_VOZ, guardar="03_voz_convertida.wav")
print(f"{len(voz_convertida) / FS:.0f} s convertidos en {time.time() - inicio:.0f} s")

escuchar(voz_original, "La voz original")
escuchar(voz_convertida, "La misma interpretación, con tu timbre")

Compara la salida con la voz original. El contenido musical, el fraseo y la afinaci?n de origen se conservan en gran medida; los artefactos pueden aumentar con mezclas densas o referencias cortas.


In [ ]:
def melodia(audio, frecuencia):
    """Altura de la voz, en hercios, cada centésima de segundo. Donde no hay voz, NaN."""
    audio_16k = librosa.resample(np.asarray(audio, dtype="float32"), orig_sr=frecuencia, target_sr=16_000)
    tono = np.asarray(motor.rmvpe.infer_from_audio(audio_16k, thred=0.03), dtype="float32")
    tono[tono < 1] = np.nan
    return tono


tono_cancion = melodia(voz_original, FS)
tono_mi_voz = melodia(mi_voz, fs_mi_voz)

print(f"La voz de la canción canta alrededor de {np.nanmedian(tono_cancion):.0f} Hz")
print(f"Tú hablas alrededor de {np.nanmedian(tono_mi_voz):.0f} Hz")
print(f"Diferencia: {12 * np.log2(np.nanmedian(tono_cancion) / np.nanmedian(tono_mi_voz)):+.0f} semitonos")

## Ajustar el registro

La transposici?n cambia la altura de la voz sin cambiar la referencia de timbre. Prueba peque?os desplazamientos y elige mediante escucha; una octava completa puede sonar artificial.


In [ ]:
voz_octava_abajo = convertir(TRABAJO / "02_voz_original.wav", MI_VOZ, semitonos=-12, guardar="04_voz_convertida_octava_abajo.wav")

escuchar(voz_convertida, "Con tu timbre, en el tono original")
escuchar(voz_octava_abajo, "Con tu timbre, una octava más grave")

La gr?fica compara las curvas de tono estimadas. Sirve para detectar cambios grandes, pero los segmentos sin voz y las estimaciones err?neas pueden afectar al dibujo.


In [ ]:
fig, eje = plt.subplots(figsize=(9, 3.4))
for audio, nombre, color in [(voz_original, "Voz original", "#8A93A8"),
                             (voz_convertida, "Convertida, tono original", "#0147FF"),
                             (voz_octava_abajo, "Convertida, una octava abajo", "#F59E0B")]:
    tono = melodia(audio, FS)
    eje.plot(np.arange(len(tono)) / 100, tono, linewidth=1.6 if nombre == "Voz original" else 1, color=color, label=nombre)
eje.set_yscale("log", base=2)
eje.set_yticks([110, 165, 220, 330, 440, 660], ["110", "165", "220", "330", "440", "660"])
eje.set_xlabel("segundos")
eje.set_ylabel("altura (Hz)")
eje.set_title("La melodía es la misma: solo cambia el timbre, y la octava si se lo pides", loc="left", fontweight="bold", fontsize=10)
eje.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.2))
eje.spines[["top", "right"]].set_visible(False)
plt.show()

Escoge la variante que suene m?s natural y as?gnala a `voz_elegida`. El ejercicio final permite comparar versiones; utiliza solo voces autorizadas.


In [ ]:
voz_elegida = voz_convertida          # o voz_convertida, si te suena mejor en el tono original

## Mezclar las pistas

Se combina la voz convertida con el instrumental extra?do. Ajusta `volumen_voz` para equilibrar ambas pistas y escucha el archivo completo antes de exportarlo.


In [ ]:
def mezclar(voz, instrumental, volumen_voz=1.0, guardar=None):
    """Suma la voz (mono) y el instrumental (estéreo), igualando antes el nivel de la voz al de la original."""
    n = min(len(voz), instrumental.shape[1], len(voz_original))
    nivel = np.sqrt((voz_original[:n] ** 2).mean()) / np.sqrt((voz[:n] ** 2).mean())
    mezcla = instrumental[:, :n] + volumen_voz * nivel * voz[:n]
    mezcla = mezcla / max(1.0, np.abs(mezcla).max() / 0.98)          # sin saturar
    if guardar:
        sf.write(TRABAJO / guardar, mezcla.T, FS)
    return mezcla


resultado = mezclar(voz_elegida, instrumental, guardar="05_cancion_con_mi_voz.wav")

escuchar(cancion, "La canción original")
escuchar(resultado, "La canción, con tu voz")

El resultado se guarda en `audios_canto/05_cancion_con_mi_voz.wav`. Esa carpeta contiene datos personales o derivados y est? excluida del control de versiones.


In [ ]:
def huella(audio, frecuencia):
    """Vector que resume el timbre de una voz, calculado con el codificador de hablante de Seed-VC."""
    audio_16k = torch.tensor(librosa.resample(np.asarray(audio, dtype="float32"), orig_sr=frecuencia, target_sr=16_000))
    rasgos = torchaudio.compliance.kaldi.fbank(audio_16k.unsqueeze(0).to(motor.device), num_mel_bins=80, dither=0, sample_frequency=16_000)
    rasgos = rasgos - rasgos.mean(dim=0, keepdim=True)
    with torch.no_grad():
        vector = motor.campplus_model(rasgos.unsqueeze(0))[0].float().cpu().numpy()
    return vector / np.linalg.norm(vector)


def diferencia_de_melodia(audio):
    """Mediana y dispersión, en semitonos, de la diferencia de altura con la voz original."""
    a, b = melodia(audio, FS), tono_cancion
    n = min(len(a), len(b))
    semitonos = 12 * np.log2(a[:n] / b[:n])
    return np.nanmedian(semitonos), np.nanstd(semitonos)


tu_timbre = huella(mi_voz, fs_mi_voz)

print("                                  parecido con tu voz    melodía frente a la original")
print(f"  Voz original                          {float(tu_timbre @ huella(voz_original, FS)):5.2f}                      —")
for nombre, audio in [("Convertida, tono original", voz_convertida), ("Convertida, octava abajo", voz_octava_abajo)]:
    mediana, dispersion = diferencia_de_melodia(audio)
    print(f"  {nombre:28}          {float(tu_timbre @ huella(audio, FS)):5.2f}            {mediana:+5.1f} semitonos (±{dispersion:.1f})")

## Qu? medir y qu? no

Las medidas de tono y timbre ayudan a comparar entradas y salidas, pero no certifican autor?a, identidad ni calidad art?stica. Interpreta los valores junto con la escucha.


In [ ]:
MI_CANTO = Path("mi_canto.wav")
OTRA_VOZ = Path("otra_voz.wav")
GRABAR_CANTO = False
SEGUNDOS_DE_CANTO = 12

if GRABAR_CANTO:
    import sounddevice as sd

    frecuencia = int(sd.query_devices(kind="input")["default_samplerate"])
    for cuenta in ["3", "2", "1"]:
        print(cuenta, end=" … ", flush=True)
        time.sleep(1)
    print("¡CANTA!")
    grabacion = sd.rec(int(SEGUNDOS_DE_CANTO * frecuencia), samplerate=frecuencia, channels=1, dtype="float32")
    sd.wait()
    sf.write(MI_CANTO, grabacion[:, 0], frecuencia)
    print("Guardado en", MI_CANTO)

if MI_CANTO.exists() and OTRA_VOZ.exists():
    mi_canto, _ = librosa.load(MI_CANTO, sr=FS, mono=True)
    con_otra_voz = convertir(MI_CANTO, OTRA_VOZ, guardar="06_mi_canto_con_otra_voz.wav")
    escuchar(mi_canto, "Tú, cantando")
    escuchar(con_otra_voz, "Lo mismo, con la otra voz")
else:
    print("Para este paso hacen falta mi_canto.wav y otra_voz.wav junto al cuaderno.")

## Uso responsable y l?mites

No imites a alguien sin permiso ni publiques audio enga?oso. Seed-VC y Demucs son proyectos de terceros: consulta sus licencias y condiciones antes de usar o redistribuir c?digo, modelos o resultados.
